In [ ]:
#!uv add langchain langchain_openai

### LCEL 개요

LCEL은 기존의 체인을 Python 코드로 복잡하게 구성하던 방식에서 벗어나, |(파이프) 연산자를 사용해 데이터 흐름을 쉽게 구성할 수 있게 해줍니다. 유닉스 파이프라인처럼 작동

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser       # str로 return

```
template ="{product} 제품 홍보문구 작성해줘"
prompt = PromptTemplate( input_variables=['product'], template=template )

gpt =  ChatOpenAI( model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
rst =gpt.invoke( prompt.format(product="컴퓨터") )
```

In [4]:
template ="{product} 제품 홍보문구 작성해줘"
prompt = PromptTemplate( input_variables=['product'], template=template )

gpt = ChatOpenAI( model="gpt-4o-mini", temperature=0)
chain = prompt | gpt
rst = chain.invoke( {'product': '컴퓨터'} )
print(rst.content)

# rst =gpt.invoke( prompt.format(product="컴퓨터") )
# print( rst.content)

물론입니다! 아래는 컴퓨터 제품을 홍보하기 위한 몇 가지 문구입니다. 필요에 따라 수정하거나 조합해 사용하실 수 있습니다.

1. **"최신 기술로 무장한 당신의 컴퓨터! 성능과 디자인을 모두 갖춘 완벽한 선택!"**

2. **"게임, 작업, 창작 – 모든 것을 완벽하게! 우리의 컴퓨터로 새로운 차원의 경험을 느껴보세요!"**

3. **"빠른 속도, 넉넉한 저장공간, 그리고 세련된 디자인! 당신의 모든 요구를 충족시키는 컴퓨터!"**

4. **"일상에서의 생산성을 높여주는 스마트한 선택! 지금 바로 만나보세요!"**

5. **"차세대 프로세서와 그래픽 카드로 무장한 컴퓨터! 당신의 꿈을 현실로 만들어 드립니다!"**

6. **"어디서나, 언제나! 휴대성과 성능을 모두 갖춘 컴퓨터로 자유롭게 작업하세요!"**

7. **"당신의 창의력을 자극하는 완벽한 파트너! 혁신적인 기술로 무장한 컴퓨터!"**

8. **"고성능, 저소음! 쾌적한 작업 환경을 제공하는 우리의 컴퓨터를 경험해보세요!"**

필요한 스타일이나 특정 기능에 맞춰 추가적인 문구를 원하시면 말씀해 주세요!


In [ ]:
template ="{product} 제품 홍보문구 작성해줘"
prompt = PromptTemplate( input_variables=['product'], template=template )

outputParser = StrOutputParser()

gpt = ChatOpenAI( model="gpt-4o-mini", temperature=0)
# chain = prompt | gpt | outputParser
chain = prompt | gpt | StrOutputParser()
rst = chain.invoke( {'product': '마우스'} )
print(rst)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Country(BaseModel):
    continet: str = Field(description="사용자가 물어본 나라가 속한 대륙")
    population: int = Field(description="사용자가 물어본 나라의 인구")

parser = JsonOutputParser(pydantic_object=Country)
format_instructions = parser.get_format_instructions()


template = "answer the question.\n{question}\n\n{format_instructions}"
prompt = PromptTemplate(   template=template,
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions}
)

gpt = ChatOpenAI( model="gpt-4o-mini", temperature=0)
chain = prompt | gpt | parser
rst = chain.invoke( {'question': "아르헨티나는 어떤 나라야?"} )
print(rst)

# rst = gpt.invoke( prompt.format( question="아르헨티나는 어떤 나라야?" ) )
# print( rst.content )

##3분 퀴즈: "비트코인은 언제 개발됐어?" lcel 을 이용하여 답변을 얻으시요( 출력형식:DatetimeOutputParser)

LangChain PipeLine 순서 : prompt -> LLM -> Parser (Output)

In [1]:
from datetime import datetime
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. 출력 결과 형태(Schema) 정의
class HistoricalEvent(BaseModel):
    event: str = Field(description="이벤트/사건 이름")
    year: int = Field(description="개발/발생 연도")
    details: str = Field(description="사건에 대한 상세 설명")

# 2. 파서 생성 및 포맷 지시문 준비
parser = PydanticOutputParser(pydantic_object=HistoricalEvent)

# 3. 프롬프트 템플릿 생성 (from_template 권장)
template = """Answer the users question:
{question}

{format_instructions}
"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. 모델 선언
gpt = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 5. LCEL 체인 결합 (Prompt -> LLM -> Parser)
chain = prompt | gpt | parser

# 6. 실행 및 결과 출력
result = chain.invoke({"question": "비트코인은 언제 개발되었나요?"})

print("파싱 결과 타입:", type(result))
print('이벤트:', result.event)
print("연도:", result.year)
print("상세 설명:", result.details)

OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************t7oA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

##  4. Q&A 챗봇 실습 (LCEL)

앞서 배운 **프롬프트 설계**와 **LCEL 파이프라인 구성법**을 직접 활용하여  
“질문 → 답변” 형태의 **Q&A 챗봇**을 완성해보세요.

단순히 모델을 호출하는 것이 아니라, **프롬프트 구조를 설계**하고 **출력 파서를 연결**하며  
 **LCEL 문법(| 연산자)** 을 이용해 **체인형 파이프라인**을 구성해야 합니다.

##### 실습 시나리오
> 당신은 **AI Q&A 챗봇 개발자**입니다.  
> 사용자가 “기술 관련 질문”을 하면, 챗봇은 **요약 + 예시 + 비유 설명**을 포함한 답변을 하도록 만들어야 합니다.  
> (예: “REST API란?”, “클라우드 컴퓨팅이 뭐야?”, “RAG는 어떤 구조야?” 등)

즉, 단순한 정의를 넘어 “무엇인지 + 왜 중요한지 + 예시” 를 한 번에 설명하는 챗봇을 설계하세요.

**과제 요구사항**

1. **`ChatPromptTemplate`** 을 사용해 아래 구조의 프롬프트를 직접 설계하세요.
   - `system`: 챗봇의 역할 정의  
     (예: “너는 기술 개념을 쉬운 비유와 예시를 섞어서 설명하는 AI 선생님이야.”)
   - `user`: 사용자의 질문을 `{question}` 변수로 전달

2. **`ChatOpenAI`** 모델을 연결하세요.  
   - 모델: `"gpt-5-nano"`  
   - `temperature=0.7` (조금 창의적인 답변 유도)

3. **`StrOutputParser`** 를 사용하여 결과를 문자열로 출력하세요.

4. **LCEL 문법**으로 체인을 완성하세요.  
   예:  
   ```python
   chain = prompt | llm | parser
   ```
5. 실행해보고 답변이 지나치게 딱딱하거나 비유나 예시가 부족하면 Prompt를 직접 수정하면서 성능을 조정해보세요.

**질문 답변 예시:**  

질문 : REST API란?  

답변 :   

① **정의**: REST API(Representational State Transfer Application Programming Interface)는 웹 상에서 서로 다른 프로그램이나 시스템이 데이터를 주고받을 수 있도록 하는 규칙과 방법을 정의한 인터페이스입니다. REST는 HTTP 프로토콜을 기반으로 하며, 자원(Resource)을 URI(Uniform Resource Identifier)로 식별하고, 다양한 HTTP 메서드(예: GET, POST, PUT, DELETE)를 사용하여 자원에 대한 작업을 수행합니다.

② **이유(중요성)**: REST API는 서로 다른 시스템 간의 통신을 쉽고 효율적으로 만들어줍니다. 이를 통해 개발자들은 복잡한 시스템을 간단하게 연결하고, 데이터를 주고받을 수 있습니다. REST API는 웹 기반 애플리케이션에서 표준으로 자리잡고 있어, 다양한 서비스를 통합하고 확장하는 데 매우 중요합니다.

③ **쉬운 예시**: REST API를 설명하기 위해 레스토랑을 비유로 들어볼 수 있습니다. 레스토랑의 메뉴판은 고객이 어떤 음식을 주문할 수 있는지를 보여주는 역할을 합니다. 고객(클라이언트)은 메뉴판(URI)을 보고 원하는 음식을 선택하고, 주문(HTTP 메서드)을 합니다. 주방(서버)은 고객의 주문을 받아 음식을 준비하고, 다시 고객에게 서빙합니다. 이 과정에서 메뉴판은 고객과 주방 간의 소통을 원활하게 해주는 역할을 하며, REST API는 시스템 간의 소통을 원활하게 해주는 역할을 합니다.


### 여기에 과제 요구 사항을 작성하세요
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

<details>
<summary>정답 보기</summary>

```python 
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7,  max_tokens=200)

# 프롬프트 설계
# system: 챗봇의 역할 정의
# user: {question} 변수를 통해 질문 전달
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 기술 개념을 알기 쉽게 설명하는 AI 선생님이야. "
     "답변은 반드시 ① 정의 ② 이유(중요성) ③ 쉬운 예시를 포함해야 해. "
     "필요하다면 비유를 사용해도 좋아."),
    ("user", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성
chain = prompt | llm | parser

# 실행 테스트
question = "REST API란?"
answer = chain.invoke({"question": question})

print(f"🧠 질문: {question}\n")
print(f"💬 답변:\n{answer}")
```
</details>

In [ ]:
 
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7,  max_tokens=200)

# 프롬프트 설계
# system: 챗봇의 역할 정의
# user: {question} 변수를 통해 질문 전달
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 기술 개념을 알기 쉽게 설명하는 AI 선생님이야. "
     "답변은 반드시 ① 정의 ② 이유(중요성) ③ 쉬운 예시를 포함해야 해. "
     "필요하다면 비유를 사용해도 좋아."),
    ("user", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성
chain = prompt | llm | parser

# 실행 테스트
question = "REST API란?"
answer = chain.invoke({"question": question})

print(f"🧠 질문: {question}\n")
print(f"💬 답변:\n{answer}")


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatOpenAI
from langchain_